In [ ]:
# eval_curve_prepare.ipynb
# Prepares the curve stage: select champions FROM grid_metrics.json (produced
# by eval_champions.ipynb, streamed while the battery runs). Pure file->file
# selection: no GPU, instant, re-tune TOP_K freely.
REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
OUT_DIR = "VICReg_review/heads/cloud_full_sweep_a100"

TOP_K = 50   # per-metric cut before intersecting; intersection size <= TOP_K

print('repo :', REPO)
print('out  :', OUT_DIR)
print('top-k:', TOP_K)


In [ ]:
# FORCE-sync to origin/main (pod repo is a mirror; untracked files untouched).
import os

if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}

%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD


In [ ]:
# Select champions: read grid_metrics.json ONLY, rank per metric, intersect
# the per-metric top-K sets, write champions.json (scores + ranks + provenance).
import json, math, os, socket, time
from pathlib import Path

metrics_path = Path(REPO) / OUT_DIR / 'grid_metrics.json'
metrics_payload = json.loads(metrics_path.read_text(encoding='utf-8'))
rows = [dict(r) for r in metrics_payload['rows']]

METRICS = {          # name -> (row key, higher_is_better)
    'tag_f1': ('tag_f1', True),
    'identity': ('identity_hit_at_1', True),
    'variant_drop': ('variant_drop_mean', False),
    'selectivity': ('selectivity', True),
}

def _ok(v):
    return v is not None and not (isinstance(v, float) and math.isnan(v))

top_sets, rank_maps = {}, {}
for mname, (key, hi) in METRICS.items():
    scored = [r for r in rows if _ok(r.get(key))]
    scored.sort(key=lambda r: r[key], reverse=hi)
    rank_maps[mname] = {r['combo_id']: i + 1 for i, r in enumerate(scored)}
    top_sets[mname] = [r['combo_id'] for r in scored[:TOP_K]]

champion_ids = set(r['combo_id'] for r in rows)
for ids in top_sets.values():
    champion_ids &= set(ids)
for r in rows:
    r['ranks'] = {m: rank_maps[m].get(r['combo_id']) for m in METRICS}
    r['rank_sum'] = sum(v for v in r['ranks'].values() if v is not None)
champions = sorted((r for r in rows if r['combo_id'] in champion_ids),
                   key=lambda r: r['rank_sum'])

payload = {
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'source_metrics': {'path': str(metrics_path),
                       'created_at': metrics_payload.get('created_at'),
                       'git_commit': metrics_payload.get('git_commit'),
                       'pool_size': metrics_payload.get('pool_size')},
    'full_n': metrics_payload.get('full_n'),
    'top_k': TOP_K,
    'selection': 'intersection of per-metric top-K over all done full-n combos '
                 '(both arms in one pool; selectivity normalises the arm difference)',
    'metrics': {m: {'key': k, 'higher_is_better': hi} for m, (k, hi) in METRICS.items()},
    'top_sets': top_sets,
    'champions': champions,
}
out_path = Path(REPO) / OUT_DIR / 'champions.json'
tmp = out_path.with_name(f'{out_path.name}.tmp.{socket.gethostname()}.{os.getpid()}')
tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=1), encoding='utf-8')
tmp.replace(out_path)

print(f'pool: {len(rows)} combos from grid_metrics.json '
      f'({len(metrics_payload.get("missing_reports", []))} not evaluated yet)')
for m in METRICS:
    print(f'  {m:13} ranked {len(rank_maps[m])} combos, top-{TOP_K} kept')
print(f'champions (intersection): {len(champions)} -> {out_path}')
print()
print(f"{'combo_id':44} {'tagF1':>6} {'id@1':>5} {'vdrop':>6} {'select':>6} {'ranks':>16}")
for r in champions:
    print(f"{r['combo_id']:44} {r['tag_f1'] or float('nan'):6.3f} "
          f"{r['identity_hit_at_1'] or float('nan'):5.2f} "
          f"{r['variant_drop_mean']:6.3f} {r['selectivity'] or float('nan'):6.3f} "
          f"{str([r['ranks'][m] for m in METRICS]):>16}")
